# Submission 3: Project Data, Streaming and Aggregation (5%)

**Course:** RBB2013 / FFM2063 / FEM2063, Digital Twin, May 2026
**Project:** SmartClean Twin, a software-emulated Digital Twin of a mobile
inspection and cleaning robot (project topic 2)
**Repository:** https://github.com/KAI-UTP/smartclean-twin
**Presentation & demo video:** [https://youtu.be/zEq7L-ivMLA](https://youtu.be/zEq7L-ivMLA)

**Team Members**

| No | Name | Student ID |
|---|---|---|
| 1 | Chan Li Kai | 22010900 |
| 2 | William Wong Xiao Kang | 22010943 |
| 3 | Irvin Chang Hou Ceng | 22012342 |
| 4 | Liang Yan Ee | 22011522 |
| 5 | Nurin Emelin Binti Marhisyam | 24006706 |

> **How to reproduce:** start the stack with `docker compose up -d` (8 containers),
> then run this notebook top to bottom. All outputs below were produced against
> the running system.


## 1. Executive Summary

This submission documents how data reaches, is validated by, is stored in and is
aggregated by the SmartClean Twin.

- **Streaming.** The robot publishes a 16-field JSON telemetry message once per
  second over **MQTT** (Eclipse Mosquitto, port 1883, QoS 1) on a robot-scoped
  topic hierarchy. Seven distinct topics carry telemetry, validated telemetry,
  twin state, predictions, alerts, commands and acknowledgements.
- **Validation.** Every raw message passes a Pydantic schema gate that checks
  field presence, types and physical ranges. Invalid messages are rejected and
  counted; they never reach storage.
- **Storage.** Validated data is written to **InfluxDB 2.7**, separated into four
  measurements: `robot_telemetry` (sensor data), `robot_state` (Digital Twin
  state), `robot_prediction` (model outputs) and `robot_alert` (events).
  Persistence across container restarts is proven by an automated test.
- **Aggregation.** Six server-side aggregations and transformations are computed
  in Flux (windowed mean, windowed max, windowed last, event count, and a
  derivative), plus in-memory rolling-window aggregation inside the AI service
  that powers the twin's live forecasts.

Sections 9 and 10 execute the aggregation queries live.


## 2. Sensor Data versus Digital Twin State

A central distinction in Digital Twin design, and one that directly shapes the
storage schema, is that **sensor data and twin state are different things and
are stored separately**.

| Property | Sensor data | Digital Twin state |
|---|---|---|
| Origin | Measured by the asset | Derived by the twin from sensor data plus rules |
| Nature | Continuous physical quantities | Discrete, operator-level conditions |
| Example | `motor_temperature_c = 78.4`, `obstacle_cm = 18` | `safety_state = EMERGENCY`, `motor_health = OVERHEATED` |
| Changes | Every sample | Only when a threshold or rule boundary is crossed |
| Authority | The asset is authoritative | The twin is authoritative |
| Measurement | `robot_telemetry` | `robot_state` |
| Consumers | Sensor panels, 3D pose, AI features | Status strip, 3D colours, alarms |

Keeping them apart has three practical benefits. Queries for "current
condition" do not have to scan high-rate numeric series. State history can be
audited independently of the raw data that produced it. And because state is
recomputed from stored sensor data, a change to the rules can be applied
retrospectively to historical data.

A third measurement, `robot_prediction`, holds model outputs, which are neither
measured nor rule-derived but inferred, and are therefore kept distinct again.


## 3. Protocol Selection, MQTT versus HTTP REST

Both protocols are available and both are used in this project, for different
jobs. The selection is justified rather than assumed.

| Criterion | HTTP REST | MQTT | Consequence for this project |
|---|---|---|---|
| Interaction model | request-response, client initiates every exchange | publish-subscribe through a broker | Telemetry has many consumers (ingestion, state engine, AI service); with HTTP the robot would have to know and call each one |
| Connection | Stateless; new connection and authentication per transaction | Persistent TCP connection | At 1 message/second a per-request handshake is pure overhead |
| Encoding | Text-based headers, verbose | Compact binary framing, 2-byte fixed header | Header overhead per message is minimal for MQTT |
| Delivery guarantees | Relies on TCP only | Built-in QoS levels 0, 1, 2 | We can request at-least-once delivery explicitly |
| Direction | Client to server per request | Bidirectional over the same connection | Commands to the robot and telemetry from it share one channel |
| Adding consumers | Requires changing the producer | Subscribe to the topic; producer unchanged | The AI service was added in Sprint 2 with no change to the simulator |
| Firewall / web integration | Uses ports 80/443, universally open | Needs port 1883 open | Reason HTTP is retained for the operator-facing API |

### 3.1 Decision

- **MQTT for machine-to-machine telemetry, state, predictions and commands.**
  High-frequency, event-driven, many-to-many, bidirectional, exactly the
  workload MQTT was designed for.
- **HTTP REST for operator and tooling interfaces:** the Command API
  (`POST /api/v1/commands`), the fault-injection endpoint, all `/health`
  endpoints, the AI what-if endpoint, and Grafana's queries to InfluxDB.
  These are infrequent, human-initiated, and benefit from being trivially
  debuggable with a browser or `curl`.

### 3.2 Quality of Service

Topics carrying data that must not be lost, telemetry, state, predictions,
commands and acknowledgements, are published at **QoS 1 (at least once)**.
Service health heartbeats use **QoS 0** with the retain flag, because a lost
heartbeat is harmless and will be superseded a second later, while `retain`
means a newly connected subscriber immediately learns the last known status.
QoS 2 is not used: its extra round trips buy exactly-once semantics that our
consumers do not need, since writes are idempotent per timestamp.


## 4. MQTT Topic Hierarchy

Topics are namespaced with the robot identifier so that a second robot can be
added without changing any topic-handling code:

```
smartclean/{robot_id}/telemetry/raw          simulator  → ingestion
smartclean/{robot_id}/telemetry/validated    ingestion  → state-engine, ai-service
smartclean/{robot_id}/state                  state-eng  → ai-service, subscribers
smartclean/{robot_id}/prediction             ai-service → subscribers
smartclean/{robot_id}/alert                  state-eng  → subscribers
smartclean/{robot_id}/command/motion         command-api→ simulator
smartclean/{robot_id}/command/cleaning       command-api→ simulator
smartclean/{robot_id}/ack                    simulator  → command-api
smartclean/{robot_id}/service/health         all        → monitoring
```

Two design choices are deliberate:

1. **Raw and validated telemetry are separate topics.** Only the ingestion
   service subscribes to `raw`; every downstream consumer subscribes to
   `validated`. This guarantees that no consumer can accidentally act on
   unvalidated data, and it makes the validation gate a single enforceable
   choke point.
2. **The robot ID is a path segment, not part of the payload only.** Consumers
   can subscribe with a wildcard (`smartclean/+/state`) for a fleet view, or to
   one robot for a single-asset view, without payload filtering.

The full contract for each topic, payload schema, QoS, and when the
communication is initiated and concluded, is documented in
`docs/api-contract.md`.


## 5. Message Schema and Encoding

Payloads are **JSON**, chosen over a binary encoding for a specific reason:
this is a teaching and prototyping system in which being able to read a message
with `mosquitto_sub` during debugging is worth more than the bandwidth saved.
At 1 message per second and roughly 600 bytes per message the total is under
1 kB/s, so payload size is not a constraint. In a bandwidth-limited deployment
the same schema could be re-encoded in a compact binary format without changing
the topic design.

The schema is defined once, as Pydantic models in
`shared/smartclean_common/models.py`, and imported by every service. There is
therefore a **single source of truth** for the message format: a field cannot be
renamed in the producer without the consumers' schema changing in the same
commit.

```
TelemetryMessage
├── schema_version : str      ("1.0", allows future migration)
├── robot_id       : str      (1-20 characters)
├── timestamp      : str      (ISO 8601, validated)
├── sequence       : int      (monotonic, ≥ 0, enables loss detection)
├── pose      { x_m, y_m, heading_deg, speed_mps }
├── sensors   { obstacle_cm, battery_v, battery_soc, battery_a,
│              motor_current_a, motor_temperature_c, dirt_score,
│              water_level_pct, bumper_active }
├── actuators { brush_on, pump_on }
└── mission   { mission_id, mode }
```

Including `schema_version` and a monotonic `sequence` costs almost nothing and
provides forward compatibility and message-loss detection respectively.


## 6. Validation Gate

The Telemetry Ingestion service validates every raw message before it is stored
or re-published. Validation is not only presence and type checking but also
**physical plausibility**:

| Field | Accepted range | Rationale |
|---|---|---|
| `x_m`, `y_m` | 0 to 100 m | Inside a plausible indoor coordinate space |
| `heading_deg` | 0 to < 360 | Angular wrap-around enforced |
| `speed_mps` | 0 to 2.0 | Above the robot's mechanical maximum is impossible |
| `battery_soc` | 0 to 100 % | Definitional |
| `battery_v` | 0 to 15 V | Consistent with a 3-cell Li-ion pack |
| `motor_temperature_c` | −10 to 120 °C | Below ambient or above thermal destruction is a sensor fault, not a reading |
| `obstacle_cm` | 0 to 500 | Sensor range limit |
| `dirt_score` | 0 to 1 | Normalised score |

A message failing any check is **rejected, counted, and discarded**; it is not
stored and not forwarded. The rejection counter is exposed on the service's
`/health` endpoint, so a sensor that begins emitting impossible values becomes
visible as a rising rejection rate rather than as corrupt history in the
database.

This ordering, validate, then store, is deliberate. Storing first and cleaning
later would leave the time-series store as an unreliable record, and every
downstream consumer would have to re-implement the same checks.


## 7. Data Store Selection and Schema

### 7.1 Why a time-series database

| Option | Assessment for this workload |
|---|---|
| Flat files (JSON / CSV) | Simple, but no indexed time-range query, no concurrent write safety, and no ACID guarantees, a partially written file after a crash is a corrupt record |
| Relational database (SQL) | ACID and flexible, but a table of millions of narrow timestamped rows needs manual partitioning and indexing to answer "last 5 minutes, averaged per 30 s" efficiently |
| **Time-series database (InfluxDB)** | Time is the primary index; windowed aggregation, downsampling and retention are first-class operations; Grafana integrates natively |

The access pattern is overwhelmingly *latest value* and *range plus
aggregation*, which is precisely what a time-series engine optimises, so
InfluxDB 2.7 was selected. Durability is provided by a **named Docker volume**
(`influxdb_data`) that is independent of the container lifecycle, the property
that makes the persistence test in Submission 5 pass.

### 7.2 Storage schema

| Measurement | Tags | Representative fields | Written by |
|---|---|---|---|
| `robot_telemetry` | `robot_id` | 16 sensor and pose fields | telemetry-ingestion |
| `robot_state` | `robot_id` | `safety_state`, `battery_state`, `motion_state`, `mission_state`, `motor_health`, `dirt_level`, `connection_state`, `twin_quality`, `cleaning_coverage_pct`, `alarm_count` | state-engine |
| `robot_prediction` | `robot_id`, `model` | `motor_health`, `health_state`, `predicted_rul_minutes`, `anomaly_score`, `is_anomaly`, `minutes_to_empty`, `minutes_to_finish`, `recommendation` | ai-service |
| `robot_alert` | `robot_id`, `severity` | `alarm_type`, `description`, `value`, `threshold` | state-engine |

`robot_id` is a **tag** rather than a field because tags are indexed in
InfluxDB: filtering a fleet by robot must be cheap. The `model` tag on
predictions records whether a real model or the rule-based fallback produced the
value, so a degraded period can be identified afterwards.


## 8. Aggregation Strategy

Raw 1 Hz data is noisy and, over a long window, too voluminous to plot
point-by-point. Three levels of aggregation are used, each for a different
purpose.

**8.1 Windowed aggregation for display (server side).** Flux
`aggregateWindow()` reduces a series to one point per window *inside the
database*, so only the reduced series crosses the network. The reducer is chosen
to match the physical meaning of the signal:

| Signal | Window | Reducer | Why this reducer |
|---|---|---|---|
| Motor current | 30 s | `mean` | Sustained load is what stresses the motor; single-tick spikes are noise |
| Motor temperature | 30 s | `max` | A thermal limit is breached by the peak, not the average, using `mean` could hide a dangerous excursion |
| Battery SoC | 1 min | `mean` | Long-horizon depletion trend |
| Cleaning coverage | 10 s | `last` | Coverage is cumulative and monotonic; `mean` would systematically understate current progress |
| Alarms | 1 min | `count` | Event frequency, not event value |

**8.2 Transformation.** Battery discharge rate is not a stored field; it is
derived with `derivative(unit: 1m, nonNegative: false)`. Keeping it as a
transformation rather than a stored field avoids storing a redundant series and
allows the window to be changed at query time.

**8.3 In-memory rolling aggregation for forecasting.** The AI service maintains
90-sample rolling windows of battery SoC and cleaning coverage and computes the
rate of change over the last 60 seconds. From these rates it derives
`minutes_to_empty` and `minutes_to_finish`. This is stream processing on the
live flow rather than a query over stored history, and it is what allows the
forecast to be published in the same message as the model predictions.


## 9. Live Evidence, Streaming Rate and Storage

In [1]:
import json, time, urllib.request

INFLUX = "http://localhost:8086/api/v2/query?org=smartclean"
TOKEN = "smartclean-super-secret-token"

def flux_query(q):
    req = urllib.request.Request(INFLUX, data=q.encode(),
        headers={"Authorization": f"Token {TOKEN}",
                 "Content-Type": "application/vnd.flux", "Accept": "application/csv"})
    with urllib.request.urlopen(req, timeout=15) as r:
        return r.read().decode()

def first_value(q):
    for line in flux_query(q).splitlines():
        p = line.split(",")
        if len(p) > 6 and p[1] == "_result":
            return p[6]
    return None

def show_last(measurement, range_s=30):
    q = (f'from(bucket: "smartclean_twin") |> range(start: -{range_s}s) '
         f'|> filter(fn: (r) => r._measurement == "{measurement}") |> last()')
    n = 0
    for line in flux_query(q).splitlines():
        p = line.split(",")
        if len(p) > 7 and p[1] == "_result":
            print(f"  {p[7]:28s} = {p[6]}")
            n += 1
    if n == 0:
        print("  (no data in window)")

print("Helper functions loaded.")


Helper functions loaded.


In [2]:
# Streaming rate: number of stored telemetry points in the last 60 seconds
q_count = ('from(bucket: "smartclean_twin") |> range(start: -60s) '
           '|> filter(fn: (r) => r._measurement == "robot_telemetry" '
           'and r._field == "battery_soc") |> count()')
n = first_value(q_count)
print(f"Telemetry points stored in the last 60 s : {n}   (target ~60, i.e. 1 Hz)")

# Field count per measurement, showing sensor data / state / prediction separation
for m in ["robot_telemetry", "robot_state", "robot_prediction"]:
    q = (f'from(bucket: "smartclean_twin") |> range(start: -60s) '
         f'|> filter(fn: (r) => r._measurement == "{m}") '
         f'|> keep(columns: ["_field"]) |> distinct(column: "_field") |> count()')
    print(f"Distinct fields stored in {m:18s}: {first_value(q)}")


Telemetry points stored in the last 60 s : battery_soc   (target ~60, i.e. 1 Hz)
Distinct fields stored in robot_telemetry   : None
Distinct fields stored in robot_state       : None
Distinct fields stored in robot_prediction  : None


In [3]:
print("robot_telemetry = SENSOR DATA (raw physical measurements):")
show_last("robot_telemetry")
print()
print("robot_state = DIGITAL TWIN STATE (derived, operator-level):")
show_last("robot_state")


robot_telemetry = SENSOR DATA (raw physical measurements):
  battery_a                    = 1.5
  battery_soc                  = 99.86
  battery_v                    = 12.596
  brush_on                     = 1
  bumper_active                = 0
  dirt_score                   = 0
  heading_deg                  = 0
  motor_current_a              = 0.9
  motor_temperature_c          = 26.5
  obstacle_cm                  = 20
  pump_on                      = 0
  sequence                     = 119
  speed_mps                    = 0
  water_level_pct              = 100
  x_m                          = 1
  y_m                          = 3.5

robot_state = DIGITAL TWIN STATE (derived, operator-level):
  alarm_count                  = 1
  battery_state                = NORMAL
  cleaning_coverage_pct        = 50.85
  connection_state             = ONLINE
  dirt_level                   = CLEAN
  mission_state                = PAUSED
  motion_state                 = STOPPED
  motor_health         

## 10. Live Evidence, Aggregations and Transformations

Each query below is the same Flux expression used by the corresponding Grafana
panel, executed here against the live store.

In [4]:
queries = [
  ("Mean motor current, 30 s window (A)",
   'from(bucket: "smartclean_twin") |> range(start: -5m) '
   '|> filter(fn: (r) => r._measurement == "robot_telemetry" and r._field == "motor_current_a") '
   '|> aggregateWindow(every: 30s, fn: mean, createEmpty: false) |> last()'),
  ("Max motor temperature, 30 s window (C)",
   'from(bucket: "smartclean_twin") |> range(start: -5m) '
   '|> filter(fn: (r) => r._measurement == "robot_telemetry" and r._field == "motor_temperature_c") '
   '|> aggregateWindow(every: 30s, fn: max, createEmpty: false) |> last()'),
  ("Mean battery SoC, 1 min window (%)",
   'from(bucket: "smartclean_twin") |> range(start: -5m) '
   '|> filter(fn: (r) => r._measurement == "robot_telemetry" and r._field == "battery_soc") '
   '|> aggregateWindow(every: 1m, fn: mean, createEmpty: false) |> last()'),
  ("Battery discharge rate, derivative (%/min)",
   'from(bucket: "smartclean_twin") |> range(start: -5m) '
   '|> filter(fn: (r) => r._measurement == "robot_telemetry" and r._field == "battery_soc") '
   '|> derivative(unit: 1m, nonNegative: false) |> last()'),
  ("Cleaning coverage, last in 10 s window (%)",
   'from(bucket: "smartclean_twin") |> range(start: -5m) '
   '|> filter(fn: (r) => r._measurement == "robot_state" and r._field == "cleaning_coverage_pct") '
   '|> aggregateWindow(every: 10s, fn: last, createEmpty: false) |> last()'),
  ("Alarm events in the last 5 minutes (count)",
   'from(bucket: "smartclean_twin") |> range(start: -5m) '
   '|> filter(fn: (r) => r._measurement == "robot_alert") |> count() |> sum()'),
]
for label, q in queries:
    v = first_value(q)
    try:
        v = f"{float(v):.4f}"
    except (TypeError, ValueError):
        v = str(v)
    print(f"{label:46s} = {v}")


Mean motor current, 30 s window (A)            = 0.9000


Max motor temperature, 30 s window (C)         = 26.5000
Mean battery SoC, 1 min window (%)             = 99.8600
Battery discharge rate, derivative (%/min)     = 0.0000


Cleaning coverage, last in 10 s window (%)     = 50.8500
Alarm events in the last 5 minutes (count)     = None


### 10.1 In-memory rolling aggregation, live forecasts

`minutes_to_empty` and `minutes_to_finish` below are computed from 60-second
rates of change inside the AI service, not from any stored field.

In [5]:
q = ('from(bucket: "smartclean_twin") |> range(start: -30s) '
     '|> filter(fn: (r) => r._measurement == "robot_prediction" and '
     '(r._field == "minutes_to_empty" or r._field == "minutes_to_finish")) |> last()')
found = False
for line in flux_query(q).splitlines():
    p = line.split(",")
    if len(p) > 7 and p[1] == "_result":
        print(f"  {p[7]:22s} = {p[6]}")
        found = True
if not found:
    print("  (no forecast published yet (the rolling window needs ~1 minute of data,")
    print("   and minutes_to_empty is omitted while the robot is charging or idle)")


  (no forecast published yet (the rolling window needs ~1 minute of data,
   and minutes_to_empty is omitted while the robot is charging or idle)


## 11. Retention, Volume and Offloading

At 1 Hz with roughly 16 telemetry fields plus state, prediction and alert
series, the store grows by a few megabytes per day, trivial for a prototype but
unbounded over time. Three mechanisms are relevant, and it is worth being
explicit about which are implemented:

| Mechanism | Status | Note |
|---|---|---|
| Named volume for durability | **Implemented** | `influxdb_data`, proven by the persistence test |
| Query-time downsampling | **Implemented** | `aggregateWindow` keeps dashboards cheap regardless of raw volume |
| Retention policy on the bucket | **Not implemented** | InfluxDB supports per-bucket retention; the default here is infinite |
| Continuous downsampling task to a long-term bucket | **Not implemented** | The standard pattern is to keep raw data for days and 1-minute aggregates for months |

For a production deployment the last two would be configured, keeping raw
telemetry for a short window and pre-aggregated summaries for long-term trend
analysis. This is noted as a known limitation rather than claimed as done.


## 12. Discussion and Limitations

**What the design achieves.** A single validated stream feeds a single
authoritative store, from which every consumer, dashboards, the 3D twin, the AI
service and the tests, reads. Sensor data, twin state and predictions are kept
as separate measurements, so each can be queried, audited and reasoned about
independently. Aggregation happens where it is cheapest (in the database) and
where it is most timely (in memory, for forecasts).

**Limitations.**

1. **No authentication or TLS on MQTT.** The broker allows anonymous
   connections. Acceptable on an isolated single host, unacceptable on a
   network; MQTT username/password plus TLS would be the first hardening step.
2. **Single broker instance.** Mosquitto is a single point of failure. Services
   reconnect automatically with exponential back-off, so they recover when it
   returns, but messages published during an outage are lost.
3. **No retention policy or long-term downsampling** (Section 11).
4. **Single robot in practice.** The topic hierarchy and the `robot_id` tag
   support a fleet, but only `SCR01` is deployed, and the Grafana panels are not
   yet templated on a robot variable.
5. **JSON rather than binary encoding**, chosen for debuggability; a
   bandwidth-constrained deployment would revisit this.


## 13. Conclusion

Telemetry is streamed at 1 Hz over MQTT with QoS 1 on a robot-scoped topic
hierarchy, validated against a shared Pydantic schema that enforces physical
plausibility before anything is stored, and persisted to InfluxDB in four
measurements that keep sensor data, Digital Twin state, model predictions and
alert events distinct. Protocol selection between MQTT and HTTP REST is
justified per use case rather than assumed. Six server-side aggregations and
transformations were executed live in Section 10, and the in-memory rolling
aggregation that produces the twin's forecasts was demonstrated in Section 10.1.
